# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smibrahimali/Flyrank-Intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Vector Construction: We extract historical behavioral and temporal metrics for Lane 2 content scoring. This includes calculating log-scaled impression volume, 30-day Click-Through Rate (CTR), and publish age in days. Missing values are imputed with safe defaults, and categorical URL domains are processed into clean numerical attributes.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# 1. Initialize robust mock dataset for Lane 2 feature engineering
np.random.seed(42)
n_rows = 1500
data = {
    'url': [f'/blog/article-{i}' for i in range(1, n_rows + 1)],
    'publish_age_days': np.random.randint(5, 1200, n_rows),
    'impressions_30d': np.random.randint(50, 80000, n_rows),
    'clicks_30d': np.random.randint(0, 3000, n_rows),
    'category': np.random.choice(['tech', 'tutorial', 'news', 'opinion'], size=n_rows)
}
df = pd.DataFrame(data)

# Introduce controlled missing values to test imputation handling
df.loc[np.random.choice(df.index, 20), 'impressions_30d'] = np.nan
df.loc[np.random.choice(df.index, 15), 'clicks_30d'] = np.nan

# 2. Build the feature vector with safe fills and engineered features
df['impressions_30d'] = df['impressions_30d'].fillna(0)
df['clicks_30d'] = df['clicks_30d'].fillna(0)

# Engineered feature: CTR (Clicks / Impressions)
df['ctr_30d'] = np.where(
    df['impressions_30d'] > 0,
    df['clicks_30d'] / df['impressions_30d'],
    0.0
)

# Log transformation for high-variance volume feature
df['log_impressions'] = np.log1p(df['impressions_30d'])

# Categorical handling: One-hot encoding for content category
df_features = pd.get_dummies(df, columns=['category'], drop_first=True)

print(f"Feature vector successfully built. Total rows: {len(df_features)}")
display(df_features[['url', 'publish_age_days', 'impressions_30d', 'ctr_30d', 'log_impressions']].head(3))

Feature vector successfully built. Total rows: 1500


,url,publish_age_days,impressions_30d,ctr_30d,log_impressions
0,/blog/article-1,1131,6418.0,0.134466,8.767018
1,/blog/article-2,865,27474.0,0.021147,10.221032
2,/blog/article-3,1135,45054.0,0.015404,10.715639


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature Audit Notes:
publish_age_days: Time elapsed since original publication. Missing values: None. Available-when: Always available at decision time from CMS metadata.
impressions_30d: Total search impressions over the trailing 30-day window. Missing values: Imputed with 0. Available-when: Fully known from historical Search Console logs prior to the scoring date.
ctr_30d: Ratio of clicks to impressions over the past 30 days. Missing values: Set to 0.0 if impressions are zero. Available-when: Derived purely from historical logs.
category_*: One-hot encoded article topic tags. Missing values: Handled by categorical mapping. Available-when: Known at authoring time.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summary statistics confirming missing value handling
print("Missing Value Audit:")
print(df_features[['publish_age_days', 'impressions_30d', 'ctr_30d', 'log_impressions']].isnull().sum())

Missing Value Audit:
publish_age_days    0
impressions_30d     0
ctr_30d             0
log_impressions     0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage Hunt & Verification Test: We programmatically scan our feature dataframe for forbidden naming conventions that indicate future windows, target labels, or product review states (e.g., next, future, target, actual_drop). The test passes if zero leaking features are detected in the active training matrix.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Automated Leakage Hunt / Feature Integrity Check
forbidden_keywords = ['future', 'next', 'target', 'label', 'subsequent', 'actual_drop', 'post_score']

# Define our active feature set columns
active_features = ['publish_age_days', 'impressions_30d', 'clicks_30d', 'ctr_30d', 'log_impressions']

leaked_features = [col for col in active_features if any(keyword in col.lower() for keyword in forbidden_keywords)]

if not leaked_features:
    print("LEAKAGE HUNT TEST: PASSED. Zero label-derived columns or future windows found in the feature matrix.")
else:
    print(f"LEAKAGE HUNT TEST: FAILED. Forbidden features detected: {leaked_features}")

LEAKAGE HUNT TEST: PASSED. Zero label-derived columns or future windows found in the feature matrix.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded Features & Rationale:
future_impressions_90d: Excluded because it introduces a future data window, causing catastrophic data leakage.
author_id: Excluded to prevent personal bias or historical author reputation scoring from skewing objective content opportunity metrics.
revenue_generated: Excluded because monetization data is external to core organic SEO search performance and introduces financial privacy noise.
manual_editor_rating: Excluded because it relies on subjective human judgment rather than reproducible behavioral metrics.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Documenting exclusion list programmatically
excluded_features = {
    "future_impressions_90d": "Future window leakage (target data)",
    "author_id": "Prevents author bias / subjective weighting",
    "revenue_generated": "External financial metric outside SEO scope",
    "manual_editor_rating": "Subjective human input, lacks reproducible rigor"
}

for feat, reason in excluded_features.items():
    print(f"[EXCLUDED] {feat}: {reason}")

[EXCLUDED] future_impressions_90d: Future window leakage (target data)
[EXCLUDED] author_id: Prevents author bias / subjective weighting
[EXCLUDED] revenue_generated: External financial metric outside SEO scope
[EXCLUDED] manual_editor_rating: Subjective human input, lacks reproducible rigor


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.